In [6]:
import pandas as pd
import numpy as np

In [7]:
# Loading cleaned wc_matches dataset
matches = pd.read_csv('../data/wc_matches.csv')

#formatting the date to proper format
matches['date'] = pd.to_datetime(matches['date'])

print(matches.shape)
print(matches.head(5))

(1298, 10)
        date      home_team    away_team  home_score  away_score  \
0 2014-01-07          Qatar       Jordan         2.0         0.0   
1 2014-01-29         Mexico  South Korea         4.0         0.0   
2 2014-02-01  United States  South Korea         2.0         0.0   
3 2014-03-05      Australia      Ecuador         3.0         4.0   
4 2014-03-05        Austria      Uruguay         1.0         1.0   

          tournament         city        country  neutral  is_friendly  
0  WAFF Championship         Doha          Qatar    False        False  
1           Friendly  San Antonio  United States     True         True  
2           Friendly       Carson  United States    False         True  
3           Friendly       London        England     True         True  
4           Friendly   Klagenfurt        Austria    False         True  


In [11]:
def get_team_stats(team, matches, last_n_years = 6):
    """
    Calculating stats for one team from recent matches
    So, cutoff date means:
    if max or the latest date is 2026-01-01
    and last_n_years = 6
    so subtracting 6 years from the latest year
    we will get 2020-01-01
    so only matches after 2020 are considered
    To keep up with the latest data we are using recent history only.
    """
    cutoff = matches['date'].max() - pd.DateOffset(years=last_n_years)

    #getting matches where the team has played home or away

    #here we are creating a df as home which consists of that team vs all other teams it has played against
    # Such as:
    #If the team = "Brazil"
    #then the home_team away_team  home_score  away_score
    #         Brazil    Argentina    2             0
    #         Brazil    France   2             0
    home = matches[(matches['home_team'] == team) & (matches['date'] >= cutoff)].copy()

    #getting all the away matches for that team only when that team has played as away_team
    #home_team away_team  home_score  away_score
    #Argentina  Brazil      2             0
    #France    Brazil   2             0
    away = matches[(matches['away_team']==team) & (matches['date']>=cutoff)].copy()

    # Calculating Home Statistics
    home_goals_scored = home['home_score'].mean()
    home_goals_conceded = home['away_score'].mean()
    home_wins = (home['home_score'] > home['away_score']).mean()

    # Calculating Away Statistics
    away_goal_scored = away['away_score'].mean()
    away_goal_conceded = away['home_score'].mean()
    away_wins = (away['away_score'] > away['home_score']).mean()

    # Total game
    total_games = len(home) + len(away)
    all_goals_scored = list(home['home_score']) + list(away['away_score'])
    all_goals_conceded = list(home['away_score']) + list(away['home_score'])

    all_goals_scored = pd.Series(all_goals_scored).dropna()
    all_goals_conceded = pd.Series(all_goals_conceded).dropna()

    # Counting total wins
    wins = (
        (home['home_score'] > home['away_score']).sum() +
        (away['away_score'] > away['home_score']).sum()
    )

    draws = (
        (home['home_score'] == home['away_score']).sum() +
        (away['away_score'] == away['home_score']).sum()
    )

    return {
        'team': team,
        'total_games' : total_games,
        'avg_goal_scored' : round(np.mean(all_goals_scored), 3),
        'avg_goal_conceded': round(np.mean(all_goals_conceded), 3),
        'win_rate': round(wins / total_games, 3) if total_games > 0 else 0,
        'draw_rate': round(draws / total_games, 3) if total_games > 0 else 0,
        'home_goals_scored' : round(home_goals_scored, 3),
        'home_goals_conceded' : round(home_goals_conceded, 3),
        'home_win_rate': round(home_wins, 3),
        'away_goals_scored': round(away_goal_scored, 3),
        'away_goals_conceded': round(away_goal_conceded, 3),
        'away_win_rate': round(away_wins, 3)
    }

#testing on a single team
print(get_team_stats("Brazil", matches))

{'team': 'Brazil', 'total_games': 47, 'avg_goal_scored': np.float64(1.727), 'avg_goal_conceded': np.float64(0.955), 'win_rate': np.float64(0.468), 'draw_rate': np.float64(0.255), 'home_goals_scored': np.float64(1.909), 'home_goals_conceded': np.float64(0.818), 'home_win_rate': np.float64(0.583), 'away_goals_scored': np.float64(1.545), 'away_goals_conceded': np.float64(1.091), 'away_win_rate': np.float64(0.348)}


In [12]:
#calculating stats for all 48 teams

world_cup_2026_teams = [

    # Hosts (CONCACAF)
    'Canada',
    'Mexico',
    'United States',

    # AFC (Asia)
    'Australia',
    'Iraq',
    'Iran',
    'Japan',
    'Jordan',
    'South Korea',
    'Qatar',
    'Saudi Arabia',
    'Uzbekistan',

    # CAF (Africa)
    'Algeria',
    'Cabo Verde',
    'Democratic Republic of Congo',
    "Côte d'Ivoire",
    'Egypt',
    'Ghana',
    'Morocco',
    'Senegal',
    'South Africa',
    'Tunisia',

    # CONCACAF
    'Curaçao',
    'Haiti',
    'Panama',

    # CONMEBOL (South America)
    'Argentina',
    'Brazil',
    'Colombia',
    'Ecuador',
    'Paraguay',
    'Uruguay',

    # OFC
    'New Zealand',

    # UEFA (Europe)
    'Austria',
    'Belgium',
    'Bosnia and Herzegovina',
    'Croatia',
    'Czechia',
    'England',
    'France',
    'Germany',
    'Netherlands',
    'Norway',
    'Portugal',
    'Scotland',
    'Spain',
    'Sweden',
    'Switzerland',
    'Türkiye'
]

all_stats = []
for team in world_cup_2026_teams:
    stats = get_team_stats(team, matches)
    all_stats.append(stats)

team_stats = pd.DataFrame(all_stats)

In [16]:
print(team_stats.head())

print(team_stats.shape)

            team  total_games  avg_goal_scored  avg_goal_conceded  win_rate  \
0         Canada           41            1.184              1.158     0.341   
1         Mexico           58            1.304              1.268     0.345   
2  United States           55            1.377              1.132     0.418   
3      Australia           31            1.207              1.103     0.419   
4           Iraq           29            0.577              1.308     0.207   

   draw_rate  home_goals_scored  home_goals_conceded  home_win_rate  \
0      0.244              1.467                0.800          0.389   
1      0.293              1.324                1.351          0.256   
2      0.218              1.628                1.116          0.489   
3      0.161              1.357                0.714          0.571   
4      0.207              0.750                1.417          0.231   

   away_goals_scored  away_goals_conceded  away_win_rate  
0              1.000                1.3

#### Adding rankings

In [19]:
rankings = pd.read_csv('../data/fifa_ranking.csv')
print(rankings.columns)
print(rankings.head(5))

Index(['rank', 'country_full', 'country_abrv', 'total_points',
       'previous_points', 'rank_change', 'confederation', 'rank_date'],
      dtype='str')
    rank       country_full country_abrv  total_points  previous_points  \
0  140.0  Brunei Darussalam          BRU           2.0              0.0   
1   33.0           Portugal          POR          38.0              0.0   
2   32.0             Zambia          ZAM          38.0              0.0   
3   31.0             Greece          GRE          38.0              0.0   
4   30.0            Algeria          ALG          39.0              0.0   

   rank_change confederation   rank_date  
0          140           AFC  1992-12-31  
1           33          UEFA  1992-12-31  
2           32           CAF  1992-12-31  
3           31          UEFA  1992-12-31  
4           30           CAF  1992-12-31  


In [20]:
latest_rankings = rankings.sort_values('rank_date').groupby('country_full').last().reset_index()

#renaming the columns to match our data
latest_rankings = latest_rankings[['country_full', 'rank', 'total_points']].rename(
    columns={
        'country_full' : 'team',
        'rank' : 'fifa_rank',
        'total_points' : 'fifa_points'
    }
)

print(latest_rankings.head())

             team  fifa_rank  fifa_points
0     Afghanistan      151.0      1034.37
1         Albania       66.0      1379.40
2         Algeria       44.0      1474.13
3  American Samoa      188.0       890.97
4         Andorra      162.0       996.05


In [21]:
team_stats = team_stats.merge(
    latest_rankings, on='team', how='left'
)

#since we are joining or merging two dataframes into one using left merging
#So it will keep all the rows from the team_stats even if the rankings are missing.
#Some team rows in the team_stats may not exist in the Fifa rankings dataset
#So filling the missing ranks by assuming weak team rank 100
team_stats['fifa_rank'] = team_stats['fifa_rank'].fillna(100)

#Same with fifa_points by assuming 0 points
team_stats['fifa_points'] = team_stats['fifa_points'].fillna(0)



In [22]:
print(team_stats.columns.to_list())

['team', 'total_games', 'avg_goal_scored', 'avg_goal_conceded', 'win_rate', 'draw_rate', 'home_goals_scored', 'home_goals_conceded', 'home_win_rate', 'away_goals_scored', 'away_goals_conceded', 'away_win_rate', 'fifa_rank', 'fifa_points']


In [25]:
print(team_stats[['team', 'avg_goal_scored', 'win_rate', 'fifa_rank']].to_string())

                            team  avg_goal_scored  win_rate  fifa_rank
0                         Canada            1.184     0.341       48.0
1                         Mexico            1.304     0.345       15.0
2                  United States            1.377     0.418      100.0
3                      Australia            1.207     0.419       23.0
4                           Iraq            0.577     0.207       55.0
5                           Iran            1.217     0.346      100.0
6                          Japan            1.639     0.487       17.0
7                         Jordan            1.065     0.294       68.0
8                    South Korea            1.500     0.417      100.0
9                          Qatar            1.036     0.226       35.0
10                  Saudi Arabia            0.606     0.229       56.0
11                    Uzbekistan            1.000     0.125       62.0
12                       Algeria            1.706     0.400       44.0
13    

#### Sanity Check - It means, does the output make logical sense? because before training ML model, we always need to verify it.

In [29]:
print("Top 10 teams by winning rate")
print(team_stats[['team', 'avg_goal_scored', 'win_rate', 'fifa_rank']].sort_values('win_rate', ascending=False).head(10))

print(" ")
print("Top 10 teams by FIFA rank")
print(team_stats[['team', 'avg_goal_scored', 'win_rate', 'fifa_rank']].sort_values('fifa_rank').head(10))

Top 10 teams by winning rate
             team  avg_goal_scored  win_rate  fifa_rank
25      Argentina            1.553     0.561        1.0
38         France            1.810     0.556        2.0
18        Morocco            1.385     0.517       12.0
6           Japan            1.639     0.487       17.0
26         Brazil            1.727     0.468        4.0
42       Portugal            1.829     0.459        6.0
3       Australia            1.207     0.419       23.0
2   United States            1.377     0.418      100.0
8     South Korea            1.500     0.417      100.0
44          Spain            1.667     0.415        8.0
 
Top 10 teams by FIFA rank
           team  avg_goal_scored  win_rate  fifa_rank
25    Argentina            1.553     0.561        1.0
38       France            1.810     0.556        2.0
33      Belgium            1.391     0.346        3.0
26       Brazil            1.727     0.468        4.0
37      England            1.542     0.407        5.0
42 

In [30]:
# Saving team_stats
team_stats.to_csv('../data/team_stats.csv', index=False)

In [31]:
print(team_stats.shape)

(48, 14)
